# D2-01 ecoinvent import and Brightway recap

This notebook bridges the BAFU-based Brightway work from Day 1 with the `ecoinvent 3.12 cutoff` workflows needed later for `premise`, `Activity Browser`, and optional `trails`.


## Learning goals

- Switch back to the shared Paris Brightway project.
- Confirm that the Day 1 BAFU import is still available.
- Import `ecoinvent 3.12 cutoff` with the `bw2io` helper interface.
- Inspect the imported database and run one quick comparison LCA.
- Prepare the project state needed for Activity Browser Part I.


## Background references

- Revisit `D1-04` if you need a refresher on project setup, imports, and method inspection.
- The `ecoinvent` import uses the helper interface already previewed at the end of `D1-04`.


## 1) Switch to the shared project and inspect the current state

We keep using the same Brightway project across the whole course so that later notebooks can build on the Day 1 work.


In [ ]:
import bw2data as bd
import pandas as pd

project_name = 'paris-lca-course-2026'
bd.projects.set_current(project_name)

databases = sorted(bd.databases)
pd.DataFrame({'database': databases})


## 2) Import `ecoinvent 3.12 cutoff`

The cell below uses `bw2io.import_ecoinvent_release()`.

- If `EI_USERNAME` and `EI_PASSWORD` are already set in your shell, the notebook reuses them.
- Otherwise, it prompts you for the credentials once.
- If the target database already exists, the cell skips the import.


In [ ]:
import os
import getpass
import bw2io as bi

ei_version = '3.12'
system_model = 'cutoff'
source_db = f'ecoinvent-{ei_version}-{system_model}'
biosphere_name = next((name for name in bd.databases if 'biosphere' in name), None)

username = os.environ.get('EI_USERNAME') or input('ecoinvent username: ')
password = os.environ.get('EI_PASSWORD') or getpass.getpass('ecoinvent password: ')

if source_db in bd.databases:
    print(f'{source_db} already exists in project {project_name}.')
else:
    bi.import_ecoinvent_release(
        version=ei_version,
        system_model=system_model,
        username=username,
        password=password,
        biosphere_name=biosphere_name,
    )
    print(f'Imported {source_db}.')


## 3) Inspect the imported database

Use a few quick checks before moving into Activity Browser.


In [ ]:
source_db = 'ecoinvent-3.12-cutoff'
db = bd.Database(source_db)

print('Number of activities:', len(db))
print('Sample databases:', sorted(bd.databases)[:10])

preview = db.search('market group for electricity, medium voltage')[:5]
[(act['name'], act.get('location'), act.get('reference product')) for act in preview]


## Checkpoint 1

In one sentence: why do we import `ecoinvent` only on Day 2, even though we already used a Brightway project on Day 1?


In [ ]:
# Answer scaffold
checkpoint_answer = '...'
checkpoint_answer


## 4) Run one compact recap LCA on an `ecoinvent` activity

We stay with the same GWP method used on Day 1 so the comparison stays easy to interpret.


In [ ]:
import bw2calc as bc

method = ('EF v3.1', 'climate change', 'global warming potential (GWP100)')

def pick_activity(database):
    preferred_terms = [
        ('market group for electricity, medium voltage', 'RER'),
        ('market for electricity, medium voltage', 'RER'),
        ('electricity, medium voltage', None),
    ]
    for term, preferred_location in preferred_terms:
        for activity in database.search(term):
            if preferred_location is None or activity.get('location') == preferred_location:
                return activity
    return database.random()

activity = pick_activity(db)
lca = bc.LCA({activity: 1}, method)
lca.lci()
lca.lcia()

print('Chosen activity:', activity['name'])
print('Location      :', activity.get('location'))
print('Method        :', method)
print('Score         :', lca.score)


## 5) Prepare for Activity Browser Part I

Activity Browser works best once the project contains both the BAFU exercises and the imported `ecoinvent` source database.

Before switching tools, confirm that you can point to all three of these project ingredients:

- the Paris course project name
- the BAFU database imported on Day 1
- the `ecoinvent-3.12-cutoff` source database imported above


## Common mistakes

- Importing `ecoinvent` into a different Brightway project than `paris-lca-course-2026`.
- Using expired or incorrect `ecoinvent` credentials.
- Forgetting that the import can take several minutes on the first run.
- Launching Activity Browser before the import finishes writing the database.


## Recap

After this notebook, the shared Paris project should contain:

- the Day 1 BAFU material
- a working `ecoinvent-3.12-cutoff` source database
- the LCIA methods needed for later Brightway, Premise, and Activity Browser work
